In [2]:
from pathlib import Path
import re
from collections import Counter


# ============================================================
# CONFIGURATION
# ============================================================

# Current location:
#
# RAG SYS/
# └── Question_Generation/
#     └── ingestion/
#
# Question Bank:
#
# RAG SYS/
# └── Question_Generation/
#     └── knowledge_base/
#         └── questions/

QUESTION_BANK_DIR = (
    Path.cwd().parent
    / "knowledge_base"
    / "questions"
)


# ============================================================
# QUESTION HEADING PATTERN
# ============================================================

QUESTION_PATTERN = re.compile(
    r"^##\s+(Q\d+)\s*[—-]\s*(.+?)\s*$",
    re.MULTILINE
)


# ============================================================
# METADATA PATTERN
# ============================================================

METADATA_PATTERN = re.compile(
    r"^-\s*([A-Za-z_ ]+?)\s*:\s*(.*?)\s*$",
    re.MULTILINE
)


# ============================================================
# EXPECTED FIELDS
# ============================================================

REQUIRED_FIELDS = [
    "Type",
    "Track",
    "Category",
    "Topic",
    "Difficulty",
    "Experience",
    "Skills",
    "Language",
    "Duration",
    "Source",
]


# ============================================================
# HELPERS
# ============================================================

def normalize_field(field: str) -> str:
    """Normalize metadata field names."""

    return (
        field
        .strip()
        .lower()
        .replace(" ", "_")
    )


def get_metadata(content: str) -> dict[str, str]:
    """Extract metadata from Markdown content."""

    metadata = {}

    for match in METADATA_PATTERN.finditer(content):

        field = normalize_field(
            match.group(1)
        )

        value = match.group(2).strip()

        metadata[field] = value

    return metadata


def get_questions(content: str) -> list[tuple[str, str]]:
    """Extract question IDs and titles."""

    return QUESTION_PATTERN.findall(content)


def is_hidden_path(path: Path) -> bool:
    """Check whether a path contains a hidden directory."""

    return any(
        part.startswith(".")
        for part in path.parts
    )


# ============================================================
# INSPECT ONE FILE
# ============================================================

def inspect_file(file_path: Path):

    content = file_path.read_text(
        encoding="utf-8"
    )

    questions = get_questions(content)

    problems = []

    # --------------------------------------------------------
    # No questions
    # --------------------------------------------------------

    if not questions:

        problems.append(
            "No questions found"
        )

        return {
            "questions": 0,
            "problems": problems,
            "metadata": []
        }


    # --------------------------------------------------------
    # Split questions
    # --------------------------------------------------------

    sections = re.split(
        r"(?=^##\s+Q\d+\s*[—-])",
        content,
        flags=re.MULTILINE
    )


    metadata_list = []


    for section in sections:

        section = section.strip()

        if not section:
            continue


        heading_match = QUESTION_PATTERN.search(
            section
        )

        if not heading_match:
            continue


        question_id = heading_match.group(1)

        title = heading_match.group(2)


        metadata = get_metadata(
            section
        )

        metadata_list.append(
            metadata
        )


        # ----------------------------------------------------
        # Check required fields
        # ----------------------------------------------------

        for field in REQUIRED_FIELDS:

            normalized = normalize_field(
                field
            )

            if not metadata.get(
                normalized
            ):

                problems.append(
                    f"{question_id}: "
                    f"missing {field}"
                )


        # ----------------------------------------------------
        # Check Question section
        # ----------------------------------------------------

        question_match = re.search(
            r"###\s*Question\s*\n+(.*?)(?=\n###|\Z)",
            section,
            re.DOTALL | re.IGNORECASE
        )

        if not question_match:

            problems.append(
                f"{question_id}: "
                f"missing Question section"
            )

        else:

            question_text = (
                question_match
                .group(1)
                .strip()
            )

            if not question_text:

                problems.append(
                    f"{question_id}: "
                    f"empty Question section"
                )


        # ----------------------------------------------------
        # Check Expected Concepts
        # ----------------------------------------------------

        concepts_match = re.search(
            r"###\s*Expected Concepts\s*\n+(.*?)(?=\n###|\Z)",
            section,
            re.DOTALL | re.IGNORECASE
        )

        if not concepts_match:

            problems.append(
                f"{question_id}: "
                f"missing Expected Concepts"
            )

        else:

            concepts = [
                line.strip()
                for line in
                concepts_match
                .group(1)
                .splitlines()
                if line.strip().startswith("-")
            ]

            if not concepts:

                problems.append(
                    f"{question_id}: "
                    f"no Expected Concepts"
                )


    return {
        "questions": len(questions),
        "problems": problems,
        "metadata": metadata_list
    }


# ============================================================
# INSPECT QUESTION BANK
# ============================================================

def inspect_question_bank():

    print("=" * 70)
    print("QUESTION BANK INSPECTION")
    print("=" * 70)


    # --------------------------------------------------------
    # Path information
    # --------------------------------------------------------

    print("\nCurrent directory:")
    print(Path.cwd())


    print("\nQuestion Bank:")
    print(QUESTION_BANK_DIR.resolve())


    print("\nQuestion Bank exists:")
    print(QUESTION_BANK_DIR.exists())


    if not QUESTION_BANK_DIR.exists():

        print(
            "\n❌ Question Bank directory "
            "does not exist."
        )

        return


    # --------------------------------------------------------
    # Find Markdown files
    # --------------------------------------------------------

    markdown_files = sorted(
        QUESTION_BANK_DIR.rglob("*.md")
    )


    visible_files = [
        file
        for file in markdown_files
        if not is_hidden_path(
            file.relative_to(
                QUESTION_BANK_DIR
            )
        )
    ]


    print(
        f"\nMarkdown files found: "
        f"{len(visible_files)}"
    )


    # --------------------------------------------------------
    # Statistics
    # --------------------------------------------------------

    total_questions = 0

    files_with_problems = 0

    all_problems = []

    track_counter = Counter()

    category_counter = Counter()

    difficulty_counter = Counter()


    # ========================================================
    # INSPECT FILES
    # ========================================================

    for file_path in visible_files:

        relative_path = file_path.relative_to(
            QUESTION_BANK_DIR
        )


        try:

            result = inspect_file(
                file_path
            )


            question_count = result[
                "questions"
            ]

            problems = result[
                "problems"
            ]

            metadata_list = result[
                "metadata"
            ]


            total_questions += (
                question_count
            )


            # ------------------------------------------------
            # Count metadata
            # ------------------------------------------------

            for metadata in metadata_list:

                track = metadata.get(
                    "track",
                    ""
                )

                category = metadata.get(
                    "category",
                    ""
                )

                difficulty = metadata.get(
                    "difficulty",
                    ""
                )


                if track:
                    track_counter[
                        track
                    ] += 1


                if category:
                    category_counter[
                        category
                    ] += 1


                if difficulty:
                    difficulty_counter[
                        difficulty
                    ] += 1


            # ------------------------------------------------
            # Print file result
            # ------------------------------------------------

            if problems:

                files_with_problems += 1

                print(
                    f"⚠ {relative_path}"
                    f" → {question_count} questions"
                    f" → {len(problems)} problems"
                )

                all_problems.extend(
                    [
                        (
                            str(relative_path),
                            problem
                        )
                        for problem in problems
                    ]
                )

            else:

                print(
                    f"✓ {relative_path}"
                    f" → {question_count} questions"
                )


        except Exception as error:

            files_with_problems += 1

            print(
                f"✗ {relative_path}"
                f" → ERROR: {error}"
            )

            all_problems.append(
                (
                    str(relative_path),
                    str(error)
                )
            )


    # ========================================================
    # SUMMARY
    # ========================================================

    print(
        "\n" + "=" * 70
    )

    print(
        "INSPECTION SUMMARY"
    )

    print(
        "=" * 70
    )


    print(
        f"\nMarkdown files:"
        f"        {len(visible_files)}"
    )

    print(
        f"Total questions:"
        f"        {total_questions}"
    )

    print(
        f"Files with problems:"
        f"     {files_with_problems}"
    )

    print(
        f"Total problems:"
        f"            {len(all_problems)}"
    )


    # ========================================================
    # TRACK DISTRIBUTION
    # ========================================================

    print(
        "\n" + "=" * 70
    )

    print(
        "QUESTIONS BY TRACK"
    )

    print(
        "=" * 70
    )


    for track, count in sorted(
        track_counter.items()
    ):

        print(
            f"{track:<25} {count}"
        )


    # ========================================================
    # CATEGORY DISTRIBUTION
    # ========================================================

    print(
        "\n" + "=" * 70
    )

    print(
        "QUESTIONS BY CATEGORY"
    )

    print(
        "=" * 70
    )


    for category, count in sorted(
        category_counter.items()
    ):

        print(
            f"{category:<30} {count}"
        )


    # ========================================================
    # DIFFICULTY DISTRIBUTION
    # ========================================================

    print(
        "\n" + "=" * 70
    )

    print(
        "QUESTIONS BY DIFFICULTY"
    )

    print(
        "=" * 70
    )


    for difficulty, count in sorted(
        difficulty_counter.items()
    ):

        print(
            f"{difficulty:<20} {count}"
        )


    # ========================================================
    # PROBLEMS
    # ========================================================

    print(
        "\n" + "=" * 70
    )

    print(
        "PROBLEMS FOUND"
    )

    print(
        "=" * 70
    )


    if all_problems:

        for file_path, problem in all_problems:

            print(
                f"\n⚠ {file_path}"
            )

            print(
                f"   {problem}"
            )

    else:

        print(
            "None 🎉"
        )


    # ========================================================
    # FINAL STATUS
    # ========================================================

    print(
        "\n" + "=" * 70
    )

    if (
        total_questions == 2150
        and len(all_problems) == 0
    ):

        print(
            "✅ QUESTION BANK PASSED INSPECTION"
        )

        print(
            "2,150 questions are structurally valid."
        )

    else:

        print(
            "⚠ QUESTION BANK NEEDS ATTENTION"
        )

    print(
        "=" * 70
    )


# ============================================================
# RUN
# ============================================================

if __name__ == "__main__":

    inspect_question_bank()

QUESTION BANK INSPECTION

Current directory:
C:\Users\User\RAG SYS\Question_Generation\ingestion

Question Bank:
C:\Users\User\RAG SYS\Question_Generation\knowledge_base\questions

Question Bank exists:
True

Markdown files found: 40
✓ ai_ml\computer_vision.md → 50 questions
✓ ai_ml\deep_learning.md → 100 questions
✓ ai_ml\llm_rag.md → 50 questions
✓ ai_ml\machine_learning.md → 100 questions
✓ ai_ml\nlp.md → 50 questions
✓ backend\databases.md → 50 questions
✓ backend\django.md → 50 questions
✓ backend\fastapi.md → 50 questions
✓ backend\python.md → 50 questions
✓ backend\rest_api.md → 50 questions
✓ behavioral\conflict.md → 50 questions
✓ behavioral\leadership.md → 50 questions
✓ behavioral\problem_solving.md → 50 questions
✓ behavioral\teamwork.md → 50 questions
✓ cloud\cloud_basics.md → 50 questions
✓ cs_fundamentals\algorithms.md → 50 questions
✓ cs_fundamentals\data_structures.md → 50 questions
✓ cs_fundamentals\databases.md → 50 questions
✓ cs_fundamentals\networking.md → 50 ques